# 📘 AI-Based Academic Decision-Support System
## Synthetic Dataset Generation v3.0 — Expanded Edge Cases
**Project:** Academic Digital Twin with ML, Fuzzy Logic & Explainability  
**Dataset Size:** 50,000 student-week records  
**Edge Case Coverage:** 10 distinct real-world paradox types (~6.5% of dataset) | 4 Student Goals

---
> ⚠️ This dataset is **synthetically generated using domain-informed constraints**.  
> It is academically valid for ML training. No real student data is used.

## 📦 Step 1: Install & Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.stats import truncnorm
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print('✅ Libraries loaded successfully')

## 🏗️ Step 2: Configuration

In [ ]:
  # ─── CONFIGURATION ───────────────────────────────────────────────
NUM_STUDENTS   = 2500     # unique students
NUM_WEEKS      = 20       # weeks per student  →  2500 × 20 = 50,000 records
EDGE_CASE_FRAC = 0.065    # 6.5% edge cases  (stays within 6–7% limit)
NOISE_STD      = 0.04     # gaussian noise on targets
RULE_FIRE_PROB = 0.78     # probabilistic rule firing

TOTAL_RECORDS  = NUM_STUDENTS * NUM_WEEKS

# 10 edge case types — evenly split across the 6.5% budget
EDGE_CASE_TYPES = [
    'burnout',              # T1 : high study + poor sleep → poor performance
    'high_aptitude',        # T2 : low effort  → decent performance
    'external_shock',       # T3 : balanced habits + sudden high pressure
    'diminishing_returns',  # T4 : consistent studier, performance plateaus
    'productive_screen',    # T5 : high screen time + high performance
    'deadline_overload',    # T6 : perfect sleep + high pressure from deadlines
    'low_attend_high_perf', # T7 : low attendance + high performance (self-study)
    'recovery_arc',         # T8 : high pressure → sudden drop to low pressure
    'goal_mismatch',        # T9 : declared CGPA goal + near-zero study hours
    'efficiency_paradox',   # T10: very high efficiency + low total study → surprising output
]

print(f'📊 Total records        : {TOTAL_RECORDS:,}')
print(f'🔀 Edge case budget     : {int(TOTAL_RECORDS * EDGE_CASE_FRAC):,} records ({EDGE_CASE_FRAC*100:.1f}%)')
print(f'🧩 Edge case types      : {len(EDGE_CASE_TYPES)}')
print(f'📌 Per-type count (approx): {int(TOTAL_RECORDS * EDGE_CASE_FRAC / len(EDGE_CASE_TYPES)):,}')

tep 3: LAYER 1 — Base Constraints

In [3]:
def truncated_normal(mean, std, low, high, size):
    a = (low  - mean) / std
    b = (high - mean) / std
    return truncnorm.rvs(a, b, loc=mean, scale=std, size=size)


def generate_base_features(n_students, n_weeks):
    records = []

    for sid in range(n_students):

        # ── Latent personality (hidden, never a feature) ──────────
        aptitude         = np.clip(np.random.normal(0.55, 0.18), 0.1, 1.0)
        base_discipline  = np.clip(np.random.normal(0.55, 0.20), 0.1, 1.0)
        stress_tolerance = np.clip(np.random.normal(0.50, 0.20), 0.1, 1.0)
        screen_use_type  = np.random.choice(['productive','recreational'], p=[0.25, 0.75])

        goal = np.random.choice(
            ['cgpa_improvement', 'workload_balance', 'skill_building', 'placement_prep'],
            p=[0.35, 0.28, 0.17, 0.20]
        )
        # Latent: is this student in placement cycle? (affects extra_hours & screen)
        in_placement_cycle = np.random.choice([True, False], p=[0.55, 0.45]) if goal == 'placement_prep' else False

        study_drift  = 0.0
        sleep_drift  = 0.0
        screen_drift = 0.0

        for week in range(1, n_weeks + 1):

            is_exam_week = week in [8, 16]
            exam_boost   = 1.8 if is_exam_week else 1.0

            study_drift  += np.random.normal(0, 0.15)
            sleep_drift  += np.random.normal(0, 0.10)
            screen_drift += np.random.normal(0, 0.12)

            study_hours = np.clip(
                truncated_normal(3.5 + base_discipline * 2.5, 1.2, 0, 10, 1)[0]
                * exam_boost + study_drift * 0.1, 0, 10
            )
            sleep_hours = np.clip(
                truncated_normal(6.5 - (exam_boost - 1) * 0.8, 0.9, 4, 9, 1)[0]
                + sleep_drift * 0.05, 4, 9
            )
            # Placement students tend to use screen more (productively: leetcode, projects)
            screen_base = 4.5 if (goal == 'placement_prep' and in_placement_cycle) else 3.0
            screen_time = np.clip(
                truncated_normal(screen_base - base_discipline * 1.0, 1.1, 0, 8, 1)[0]
                + screen_drift * 0.08, 0, 8
            )
            attendance = np.clip(
                truncated_normal(78 + base_discipline * 12, 8, 60, 100, 1)[0],
                60, 100
            )
            deadline_count = int(np.clip(
                np.random.poisson(2.0 + (1.5 if is_exam_week else 0)), 0, 6
            ))
            late_night_ratio = np.clip(
                np.random.beta(2, 5) + (0.15 if screen_time > 5 else 0), 0, 1
            )
            study_sessions = int(np.clip(
                np.random.poisson(max(1, study_hours * 1.2)), 1, 14
            ))
            # Placement students: more extra hours (mock interviews, hackathons, projects)
            if goal == 'placement_prep' and in_placement_cycle:
                extra_hours = np.clip(np.random.uniform(3.5, 8.0), 0, 10)
            else:
                extra_hours = np.clip(np.random.exponential(1.8), 0, 10)

            records.append({
                'student_id'       : f'S{sid:04d}',
                'week'             : week,
                'goal'             : goal,
                'study_hours_day'  : round(study_hours, 2),
                'sleep_hours_day'  : round(sleep_hours, 2),
                'screen_time_day'  : round(screen_time, 2),
                'attendance_pct'   : round(attendance, 1),
                'deadline_count'   : deadline_count,
                'late_night_ratio' : round(late_night_ratio, 3),
                'study_sessions'   : study_sessions,
                'extra_hours'      : round(extra_hours, 2),
                '_aptitude'        : round(aptitude, 4),
                '_discipline'      : round(base_discipline, 4),
                '_stress_tol'      : round(stress_tolerance, 4),
                '_is_exam_week'    : int(is_exam_week),
                '_screen_use_type'      : screen_use_type,
                '_in_placement_cycle'  : int(in_placement_cycle),
            })

    return pd.DataFrame(records)


print('⏳ Generating base features...')
df = generate_base_features(NUM_STUDENTS, NUM_WEEKS)
print(f'✅ Base layer done — shape: {df.shape}')
df.head(3)

⏳ Generating base features...
✅ Base layer done — shape: (50000, 17)


,student_id,week,goal,study_hours_day,sleep_hours_day,screen_time_day,attendance_pct,deadline_count,late_night_ratio,study_sessions,extra_hours,_aptitude,_discipline,_stress_tol,_is_exam_week,_screen_use_type,_in_placement_cycle
0,S0000,1,cgpa_improvement,2.38,8.17,3.55,77.8,1,0.550,2,0.82,0.6394,0.5223,0.6295,0,productive,0
1,S0000,2,cgpa_improvement,5.13,5.02,2.80,76.5,0,0.108,3,4.32,0.6394,0.5223,0.6295,0,productive,0
2,S0000,3,cgpa_improvement,4.26,6.56,2.62,77.0,6,0.251,5,7.80,0.6394,0.5223,0.6295,0,productive,0


## 🔗 Step 4: LAYER 2 — Behavioral Correlation Features

In [4]:
def add_behavioral_features(df, rule_fire_prob=0.78):
    n     = len(df)
    fires = np.random.rand(n) < rule_fire_prob

    sleep_norm  = (df['sleep_hours_day'] - 4) / 5
    screen_norm = 1 - (df['screen_time_day'] / 8)
    session_norm= df['study_sessions'] / 14

    # Productive screen users get a bonus on efficiency
    productive_mask   = (df['_screen_use_type'] == 'productive').values
    screen_efficiency = np.where(productive_mask, screen_norm * 1.15, screen_norm)

    base_eff     = (sleep_norm * 0.35 + screen_efficiency * 0.35 + session_norm * 0.30)
    late_penalty = df['late_night_ratio'] * 0.20
    efficiency   = np.clip(base_eff - late_penalty, 0, 1)

    df['study_efficiency'] = np.where(
        fires,
        np.clip(efficiency + np.random.normal(0, 0.05, n), 0, 1),
        np.random.uniform(0.2, 0.9, n)
    ).round(3)

    df['deadline_density']  = (df['deadline_count'] / 6).round(3)

    # study_hrs_per_session: avg hours per study session (proxy for session intensity)
    # Precisely named to distinguish from a time-series variance metric.
    df['study_hrs_per_session'] = np.clip(
        df['study_hours_day'] / df['study_sessions'].replace(0, 1), 0, 5
    ).round(3)

    stability = np.clip(
        sleep_norm * 0.5 + (1 - df['study_hrs_per_session'] / 5) * 0.5, 0, 1
    )
    df['habit_stability'] = stability.round(3)

    cgpa_mask      = df['goal'] == 'cgpa_improvement'
    placement_mask = df['goal'] == 'placement_prep'
    df['effort_mismatch'] = np.where(
        cgpa_mask,
        np.clip((df['study_hours_day'] / 10) - (df['attendance_pct'] / 100), -1, 1),
        np.where(
            placement_mask,
            # Placement mismatch: high screen but low extra hours = not really prepping
            np.clip((df['screen_time_day'] / 8) - (df['extra_hours'] / 10), -1, 1),
            0.0
        )
    ).round(3)

    # Cumulative pressure from previous week (rolling behavioral stress)
    df = df.sort_values(['student_id', 'week']).reset_index(drop=True)
    df['prev_deadline_density'] = df.groupby('student_id')['deadline_density'].shift(1).fillna(0)

    print('✅ Behavioral features added')
    return df


df = add_behavioral_features(df, RULE_FIRE_PROB)
df[['student_id','week','study_efficiency','deadline_density',
    'habit_stability','effort_mismatch','prev_deadline_density']].head(4)

✅ Behavioral features added


,student_id,week,study_efficiency,deadline_density,habit_stability,effort_mismatch,prev_deadline_density
0,S0000,1,0.550,0.167,0.798,-0.540,0.000
1,S0000,2,0.376,0.000,0.431,-0.252,0.167
2,S0000,3,0.390,1.000,0.671,-0.344,0.000
3,S0000,4,0.576,0.500,0.566,-0.173,1.000


## 🎯 Step 5: LAYER 3 — Outcome Generation (Hidden Ground Truth)

In [ ]:
def generate_targets(df, noise_std=0.04):
    n = len(df)

    # ── TARGET 1: Academic Performance Score ∈ [0, 100] ──────────
    # Blended: 70% mediated by efficiency, 30% direct hours signal
    effort_score     = (df['study_hours_day'] / 10) * (df['study_efficiency'] * 0.70 + 0.30)
    deadline_penalty = df['deadline_density'] * 0.15
    screen_penalty   = (df['screen_time_day'] / 8) * np.where(
        df['_screen_use_type'] == 'productive', 0.03, 0.10  # productive screen hurts less
    )
    aptitude_bonus   = df['_aptitude'] * 0.25
    attend_bonus     = (df['attendance_pct'] / 100) * 0.15

    perf_raw = (
        0.18                   # baseline intercept — shifts distribution to realistic 50–60 range
        + effort_score * 0.45
        + aptitude_bonus
        + attend_bonus
        - deadline_penalty
        - screen_penalty
        + np.random.normal(0, noise_std, n)
    )
    df['performance_score'] = np.clip(perf_raw * 100, 0, 100).round(2)

    # ── TARGET 2: Workload Pressure Score ∈ [0, 1] ───────────────
    sleep_stress    = np.clip(1 - (df['sleep_hours_day'] - 4) / 5, 0, 1)
    screen_stress   = df['screen_time_day'] / 8
    deadline_stress = df['deadline_density']
    carryover       = df['prev_deadline_density'] * 0.10   # stress carries over
    tol_factor      = 1 - df['_stress_tol'] * 0.3

    pressure_raw = (
        deadline_stress * 0.38
        + sleep_stress   * 0.28
        + screen_stress  * 0.17
        + (1 - df['habit_stability']) * 0.10
        + carryover      * 0.07
    ) * tol_factor + np.random.normal(0, noise_std, n)

    df['pressure_score'] = np.clip(pressure_raw, 0, 1).round(3)

    df['pressure_label'] = pd.cut(
        df['pressure_score'],
        bins=[0, 0.33, 0.55, 1.0],   # lowered High threshold: 0.66 → 0.55
        labels=['Low', 'Medium', 'High'],
        include_lowest=True
    )

    # ── TARGET 3: Goal Alignment Score ∈ [0, 1] ──────────────────
    alignment = np.zeros(n)
    cgpa_m    = (df['goal'] == 'cgpa_improvement').values
    bal_m     = (df['goal'] == 'workload_balance').values
    skill_m   = (df['goal'] == 'skill_building').values
    place_m   = (df['goal'] == 'placement_prep').values

    # CGPA: high study + high attendance + manageable pressure
    alignment[cgpa_m] = np.clip(
        (df.loc[cgpa_m, 'study_hours_day'] / 10) * 0.5
        + (df.loc[cgpa_m, 'attendance_pct'] / 100) * 0.3
        + (1 - df.loc[cgpa_m, 'pressure_score']) * 0.2, 0, 1
    )
    # Balance: low pressure + stable habits + moderate study
    alignment[bal_m] = np.clip(
        (1 - df.loc[bal_m, 'pressure_score']) * 0.5
        + df.loc[bal_m, 'habit_stability'] * 0.3
        + np.clip(1 - abs(df.loc[bal_m, 'study_hours_day'] - 4) / 4, 0, 1) * 0.2, 0, 1
    )
    # Skill building: extra hours + high efficiency + low recreational screen
    alignment[skill_m] = np.clip(
        (df.loc[skill_m, 'extra_hours'] / 10) * 0.4
        + df.loc[skill_m, 'study_efficiency'] * 0.35
        + (1 - df.loc[skill_m, 'screen_time_day'] / 8) * 0.25, 0, 1
    )
    # Placement prep: high extra hours (projects/hackathons) + productive screen
    #                 + moderate pressure tolerance + decent attendance
    alignment[place_m] = np.clip(
        (df.loc[place_m, 'extra_hours'] / 10) * 0.45
        + (df.loc[place_m, 'screen_time_day'] / 8) * 0.25   # screen = productive prep
        + (df.loc[place_m, 'attendance_pct'] / 100) * 0.15
        + (1 - df.loc[place_m, 'pressure_score']) * 0.15, 0, 1
    )

    df['goal_alignment'] = np.clip(
        alignment + np.random.normal(0, noise_std, n), 0, 1
    ).round(3)

    df['alignment_label'] = pd.cut(
        df['goal_alignment'],
        bins=[0, 0.33, 0.66, 1.0],
        labels=['Misaligned', 'Partial', 'Aligned'],
        include_lowest=True
    )

    print('✅ Targets generated')
    return df


df = generate_targets(df, NOISE_STD)
df[['student_id','week','performance_score',
    'pressure_score','pressure_label',
    'goal_alignment','alignment_label']].head(5)

## 🔀 Step 6: Expanded Edge Case Injection — 10 Types

| # | Type | Real-world scenario |
|---|------|---------------------|
| T1 | burnout | High study + poor sleep → poor performance |
| T2 | high_aptitude | Low effort → decent performance |
| T3 | external_shock | Balanced habits + sudden high pressure |
| T4 | diminishing_returns | Consistent study → performance plateau |
| T5 | productive_screen | High screen + high performance (coding/research) |
| T6 | deadline_overload | Good sleep + high pressure from deadlines alone |
| T7 | low_attend_high_perf | Low attendance + high performance (self-study) |
| T8 | recovery_arc | High pressure week → sudden recovery |
| T9 | goal_mismatch | CGPA goal + near-zero study hours |
| T10 | efficiency_paradox | Very high efficiency + low hours → surprise output |

In [ ]:
def inject_edge_cases(df, frac=0.065):
    """
    Inject 10 distinct edge case types covering major real-world academic paradoxes.
    Total edge cases stay within the 6–7% budget.
    """
    n_total   = len(df)
    n_edge    = int(n_total * frac)          # e.g. 3250 records
    n_per_type= n_edge // 10                 # ~325 per type

    all_idx   = np.random.permutation(df.index.tolist())
    edge_idx  = all_idx[:n_edge]

    # Split into 10 chunks
    chunks = np.array_split(edge_idx, 10)
    t1, t2, t3, t4, t5, t6, t7, t8, t9, t10 = chunks

    # ── T1: BURNOUT ──────────────────────────────────────────────
    # Very high study + very poor sleep → poor performance, high pressure
    df.loc[t1, 'study_hours_day']   = np.random.uniform(7.5, 10.0, len(t1)).round(2)
    df.loc[t1, 'sleep_hours_day']   = np.random.uniform(4.0,  5.0, len(t1)).round(2)
    df.loc[t1, 'study_efficiency']  = np.random.uniform(0.08, 0.28, len(t1)).round(3)
    df.loc[t1, 'performance_score'] = np.random.uniform(25.0, 50.0, len(t1)).round(2)
    df.loc[t1, 'pressure_score']    = np.random.uniform(0.72, 1.00, len(t1)).round(3)
    df.loc[t1, 'goal_alignment']    = np.random.uniform(0.05, 0.30, len(t1)).round(3)

    # ── T2: HIGH APTITUDE / LOW EFFORT ───────────────────────────
    # Minimal study hours → still decent performance (high latent aptitude)
    df.loc[t2, 'study_hours_day']   = np.random.uniform(0.3,  2.5, len(t2)).round(2)
    df.loc[t2, 'study_sessions']    = np.random.randint(1, 4, len(t2))
    df.loc[t2, 'performance_score'] = np.random.uniform(60.0, 82.0, len(t2)).round(2)
    df.loc[t2, 'pressure_score']    = np.random.uniform(0.05, 0.28, len(t2)).round(3)

    # ── T3: EXTERNAL SHOCK ───────────────────────────────────────
    # Good habits week → suddenly overwhelmed (illness / family event)
    df.loc[t3, 'habit_stability']   = np.random.uniform(0.65, 0.88, len(t3)).round(3)
    df.loc[t3, 'sleep_hours_day']   = np.random.uniform(6.5,  8.5, len(t3)).round(2)
    df.loc[t3, 'deadline_count']    = np.random.randint(5, 7, len(t3))
    df.loc[t3, 'deadline_density']  = (df.loc[t3, 'deadline_count'] / 6).round(3)
    df.loc[t3, 'pressure_score']    = np.random.uniform(0.78, 1.00, len(t3)).round(3)
    df.loc[t3, 'performance_score'] = np.random.uniform(35.0, 58.0, len(t3)).round(2)

    # ── T4: DIMINISHING RETURNS ───────────────────────────────────
    # Same consistent hours each week but performance stops growing
    df.loc[t4, 'study_hours_day']   = np.random.uniform(5.5,  7.5, len(t4)).round(2)
    df.loc[t4, 'study_efficiency']  = np.random.uniform(0.30, 0.48, len(t4)).round(3)
    df.loc[t4, 'habit_stability']   = np.random.uniform(0.75, 0.95, len(t4)).round(3)
    df.loc[t4, 'performance_score'] = np.random.uniform(52.0, 65.0, len(t4)).round(2)  # plateau zone
    df.loc[t4, 'pressure_score']    = np.random.uniform(0.30, 0.55, len(t4)).round(3)

    # ── T5: PRODUCTIVE SCREEN TIME ────────────────────────────────
    # High screen time (coding / research) → high performance
    df.loc[t5, 'screen_time_day']   = np.random.uniform(5.5,  8.0, len(t5)).round(2)
    df.loc[t5, 'late_night_ratio']  = np.random.uniform(0.05, 0.20, len(t5)).round(3)  # daytime screen
    df.loc[t5, 'study_efficiency']  = np.random.uniform(0.65, 0.90, len(t5)).round(3)
    df.loc[t5, 'performance_score'] = np.random.uniform(68.0, 90.0, len(t5)).round(2)
    df.loc[t5, 'pressure_score']    = np.random.uniform(0.10, 0.38, len(t5)).round(3)

    # ── T6: DEADLINE OVERLOAD ─────────────────────────────────────
    # Great sleep, stable habits — but pure deadline density causes high pressure
    df.loc[t6, 'sleep_hours_day']   = np.random.uniform(7.0,  9.0, len(t6)).round(2)
    df.loc[t6, 'habit_stability']   = np.random.uniform(0.70, 0.90, len(t6)).round(3)
    df.loc[t6, 'deadline_count']    = np.random.randint(5, 7, len(t6))
    df.loc[t6, 'deadline_density']  = (df.loc[t6, 'deadline_count'] / 6).round(3)
    df.loc[t6, 'pressure_score']    = np.random.uniform(0.70, 0.95, len(t6)).round(3)
    df.loc[t6, 'performance_score'] = np.random.uniform(45.0, 65.0, len(t6)).round(2)

    # ── T7: LOW ATTENDANCE, HIGH PERFORMANCE ─────────────────────
    # Self-study dominant — skips class but excels through independent work
    df.loc[t7, 'attendance_pct']    = np.random.uniform(60.0, 70.0, len(t7)).round(1)
    df.loc[t7, 'study_hours_day']   = np.random.uniform(6.0,  9.0, len(t7)).round(2)
    df.loc[t7, 'study_efficiency']  = np.random.uniform(0.70, 0.92, len(t7)).round(3)
    df.loc[t7, 'performance_score'] = np.random.uniform(70.0, 90.0, len(t7)).round(2)
    df.loc[t7, 'pressure_score']    = np.random.uniform(0.15, 0.40, len(t7)).round(3)

    # ── T8: RECOVERY ARC ─────────────────────────────────────────
    # Previously high-pressure student who suddenly recovers (intervention / holiday)
    df.loc[t8, 'prev_deadline_density'] = np.random.uniform(0.7, 1.0, len(t8)).round(3)  # last week bad
    df.loc[t8, 'deadline_count']    = np.random.randint(0, 2, len(t8))    # this week light
    df.loc[t8, 'deadline_density']  = (df.loc[t8, 'deadline_count'] / 6).round(3)
    df.loc[t8, 'sleep_hours_day']   = np.random.uniform(7.5,  9.0, len(t8)).round(2)
    df.loc[t8, 'pressure_score']    = np.random.uniform(0.05, 0.22, len(t8)).round(3)  # sudden drop
    df.loc[t8, 'performance_score'] = np.random.uniform(60.0, 78.0, len(t8)).round(2)

    # ── T9: GOAL MISMATCH ─────────────────────────────────────────
    # Declared CGPA goal but behaviour shows near-zero study hours
    df.loc[t9, 'goal']              = 'cgpa_improvement'
    df.loc[t9, 'study_hours_day']   = np.random.uniform(0.0,  1.5, len(t9)).round(2)
    df.loc[t9, 'study_sessions']    = np.random.randint(0, 3, len(t9))
    df.loc[t9, 'attendance_pct']    = np.random.uniform(60.0, 72.0, len(t9)).round(1)
    df.loc[t9, 'effort_mismatch']   = np.random.uniform(0.55, 1.00, len(t9)).round(3)  # high mismatch
    df.loc[t9, 'goal_alignment']    = np.random.uniform(0.0,  0.20, len(t9)).round(3)  # very misaligned
    df.loc[t9, 'performance_score'] = np.random.uniform(20.0, 45.0, len(t9)).round(2)

    # ── T10: EFFICIENCY PARADOX ────────────────────────────────────
    # Surprisingly high efficiency with very few study hours → unexpected output
    df.loc[t10, 'study_hours_day']  = np.random.uniform(0.5,  2.0, len(t10)).round(2)
    df.loc[t10, 'study_efficiency'] = np.random.uniform(0.88, 1.00, len(t10)).round(3)
    df.loc[t10, 'sleep_hours_day']  = np.random.uniform(7.5,  9.0, len(t10)).round(2)
    df.loc[t10, 'screen_time_day']  = np.random.uniform(0.0,  1.5, len(t10)).round(2)
    df.loc[t10, 'performance_score']= np.random.uniform(65.0, 85.0, len(t10)).round(2)
    df.loc[t10, 'pressure_score']   = np.random.uniform(0.05, 0.25, len(t10)).round(3)

    # ── FIX: Re-derive BEHAVIORAL features for edge-case rows ─────
    # Raw values (study_hours_day, sleep_hours_day, deadline_count, etc.) were
    # overwritten above, so we must recalculate the derived features to stay
    # internally consistent — otherwise study_hrs_per_session, habit_stability,
    # and deadline_density would reflect the OLD raw values.
    all_edge_idx = np.concatenate([t1, t2, t3, t4, t5, t6, t7, t8, t9, t10])

    # Recalculate study_hrs_per_session (hours per study session)
    safe_sessions = df.loc[all_edge_idx, 'study_sessions'].copy()
    safe_sessions[safe_sessions == 0] = 1
    df.loc[all_edge_idx, 'study_hrs_per_session'] = np.clip(
        df.loc[all_edge_idx, 'study_hours_day'] / safe_sessions,
        0, 5
    ).round(3)

    # Recalculate habit_stability from updated sleep and study_hrs_per_session
    sleep_norm_ec = (df.loc[all_edge_idx, 'sleep_hours_day'] - 4) / 5
    df.loc[all_edge_idx, 'habit_stability'] = np.clip(
        sleep_norm_ec * 0.5 +
        (1 - df.loc[all_edge_idx, 'study_hrs_per_session'] / 5) * 0.5,
        0, 1
    ).round(3)

    # Recalculate deadline_density for edge types where deadline_count was overwritten
    for idx_chunk in [t3, t6, t8]:
        df.loc[idx_chunk, 'deadline_density'] = (df.loc[idx_chunk, 'deadline_count'] / 6).round(3)

    # Recalculate effort_mismatch for changed rows (T9 was set manually; others recalc)
    non_t9_edge = np.concatenate([t1, t2, t3, t4, t5, t6, t7, t8, t10])
    cgpa_mask_ec   = df.loc[non_t9_edge, 'goal'] == 'cgpa_improvement'
    place_mask_ec  = df.loc[non_t9_edge, 'goal'] == 'placement_prep'
    cgpa_idx  = non_t9_edge[cgpa_mask_ec.values]
    place_idx = non_t9_edge[place_mask_ec.values]
    other_idx = non_t9_edge[(~cgpa_mask_ec & ~place_mask_ec).values]
    if len(cgpa_idx):
        df.loc[cgpa_idx, 'effort_mismatch'] = np.clip(
            (df.loc[cgpa_idx, 'study_hours_day'] / 10) - (df.loc[cgpa_idx, 'attendance_pct'] / 100),
            -1, 1
        ).round(3)
    if len(place_idx):
        df.loc[place_idx, 'effort_mismatch'] = np.clip(
            (df.loc[place_idx, 'screen_time_day'] / 8) - (df.loc[place_idx, 'extra_hours'] / 10),
            -1, 1
        ).round(3)
    if len(other_idx):
        df.loc[other_idx, 'effort_mismatch'] = 0.0

    # ── Re-derive labels — cast to str to avoid NaN in categorical columns ──
    df['pressure_label'] = pd.cut(
        df['pressure_score'].astype(float), bins=[0, 0.33, 0.55, 1.0],
        labels=['Low', 'Medium', 'High'], include_lowest=True
    ).astype(str)
    df['alignment_label'] = pd.cut(
        df['goal_alignment'].astype(float), bins=[0, 0.33, 0.66, 1.0],
        labels=['Misaligned', 'Partial', 'Aligned'], include_lowest=True
    ).astype(str)

    # Tag edge case type for documentation
    df['edge_case_type'] = 'normal'
    type_names = ['burnout','high_aptitude','external_shock','diminishing_returns',
                  'productive_screen','deadline_overload','low_attend_high_perf',
                  'recovery_arc','goal_mismatch','efficiency_paradox']
    for idx_chunk, name in zip(chunks, type_names):
        df.loc[idx_chunk, 'edge_case_type'] = name

    df['is_edge_case'] = (df['edge_case_type'] != 'normal').astype(int)

    print(f'✅ Edge cases injected : {n_edge:,} records ({frac*100:.1f}%)')
    print(f'   Per-type approx    : {n_per_type:,}')
    print()
    print('📋 Edge case distribution:')
    print(df['edge_case_type'].value_counts().to_string())
    return df


df = inject_edge_cases(df, EDGE_CASE_FRAC)

## 🧹 Step 7: Prepare Final ML-Ready Dataset

In [ ]:
LATENT_COLS  = ['_aptitude', '_discipline', '_stress_tol', '_is_exam_week', '_screen_use_type', '_in_placement_cycle']

df_ml = df.drop(columns=LATENT_COLS).copy()

FEATURE_COLS = [
    'study_hours_day', 'sleep_hours_day', 'screen_time_day',
    'attendance_pct', 'deadline_count', 'late_night_ratio',
    'study_sessions', 'extra_hours',
    'study_efficiency', 'deadline_density', 'study_hrs_per_session',
    'habit_stability', 'effort_mismatch', 'prev_deadline_density'
]
TARGET_COLS = [
    'performance_score', 'pressure_score', 'pressure_label',
    'goal_alignment', 'alignment_label'
]
META_COLS = ['student_id', 'week', 'goal', 'is_edge_case', 'edge_case_type']

df_ml = df_ml[META_COLS + FEATURE_COLS + TARGET_COLS]

print(f'📐 Final dataset shape      : {df_ml.shape}')
print(f'   Feature columns          : {len(FEATURE_COLS)}  (incl. prev_deadline_density)')
print(f'   Target columns           : {len(TARGET_COLS)}')
print(f'   Unique students          : {df_ml.student_id.nunique():,}')
print(f'   Weeks per student        : {df_ml.week.nunique()}')
print(f'   Total edge case records  : {df_ml.is_edge_case.sum():,}')
print(f'   Edge case %              : {df_ml.is_edge_case.mean()*100:.2f}%')
df_ml.describe().round(2)

## 📊 Step 8: Visualizations

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle('Synthetic Dataset v3.0 — Distributions, Correlations & Edge Cases',
             fontsize=14, fontweight='bold')

sample = df_ml.sample(2000, random_state=42)

# 1. Performance distribution
axes[0,0].hist(df_ml['performance_score'], bins=60, color='steelblue', edgecolor='white', alpha=0.85)
axes[0,0].set_title('Performance Score Distribution')
axes[0,0].set_xlabel('Score (0–100)')

# 2. Pressure label distribution
df_ml['pressure_label'].value_counts().sort_index().plot(
    kind='bar', ax=axes[0,1], color=['green','orange','red'], edgecolor='white'
)
axes[0,1].set_title('Workload Pressure Label Distribution')
axes[0,1].tick_params(axis='x', rotation=0)

# 3. Study hours vs performance — edge cases highlighted
normal_s = sample[sample['is_edge_case'] == 0]
edge_s   = sample[sample['is_edge_case'] == 1]
axes[0,2].scatter(normal_s['study_hours_day'], normal_s['performance_score'],
                  alpha=0.25, s=10, color='steelblue', label='Normal')
axes[0,2].scatter(edge_s['study_hours_day'], edge_s['performance_score'],
                  alpha=0.7,  s=18, color='crimson',   label='Edge Case')
axes[0,2].set_title('Study Hours vs Performance\n(red = edge cases)')
axes[0,2].set_xlabel('Study Hours/Day')
axes[0,2].set_ylabel('Performance Score')
axes[0,2].legend(fontsize=8)

# 4. Edge case type distribution
ec_counts = df_ml[df_ml['is_edge_case']==1]['edge_case_type'].value_counts()
ec_counts.plot(kind='barh', ax=axes[1,0], color='coral', edgecolor='white')
axes[1,0].set_title('Edge Case Type Distribution')
axes[1,0].set_xlabel('Count')

# 5. Correlation heatmap
corr_cols = ['study_hours_day','sleep_hours_day','screen_time_day',
             'deadline_density','study_efficiency','habit_stability',
             'performance_score','pressure_score','goal_alignment']
corr = df_ml[corr_cols].corr()
sns.heatmap(corr, ax=axes[1,1], cmap='coolwarm', annot=True,
            fmt='.2f', annot_kws={'size':7}, linewidths=0.4)
axes[1,1].set_title('Feature Correlation Heatmap')

# 6. Weekly avg performance with exam week markers
weekly_avg = df_ml.groupby('week')['performance_score'].mean()
axes[1,2].plot(weekly_avg.index, weekly_avg.values, marker='o', color='teal', linewidth=2)
axes[1,2].axvline(x=8,  color='red', linestyle='--', alpha=0.6, label='Exam Week 1')
axes[1,2].axvline(x=16, color='red', linestyle='--', alpha=0.6, label='Exam Week 2')
axes[1,2].set_title('Avg Performance — Weekly Trend')
axes[1,2].set_xlabel('Week')
axes[1,2].set_ylabel('Avg Score')
axes[1,2].legend(fontsize=8)

plt.tight_layout()
plt.savefig('dataset_overview_v3.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: dataset_overview_v3.png')

## 💾 Step 9: Save Dataset

In [ ]:
df_ml.to_csv('academic_digital_twin_dataset_v3.csv', index=False)
df_ml.sample(500, random_state=42).to_csv('dataset_sample_500_v3.csv', index=False)
df_ml[df_ml['is_edge_case']==1].to_csv('edge_cases_only.csv', index=False)
df.to_csv('academic_dataset_with_latents_v3.csv', index=False)

print(f'✅ Full dataset          : academic_digital_twin_dataset_v3.csv  ({len(df_ml):,} rows)')
print(f'✅ Sample (500)          : dataset_sample_500_v3.csv')
print(f'✅ Edge cases only       : edge_cases_only.csv  ({df_ml.is_edge_case.sum():,} rows)')
print(f'✅ With latents          : academic_dataset_with_latents_v3.csv  (docs only)')

## ✅ Step 10: Full Sanity Check

In [ ]:
print('=' * 62)
print('  DATASET QUALITY SANITY CHECK — v3.0')
print('=' * 62)

edge_pct = df_ml['is_edge_case'].mean() * 100

checks = [
    ('Total records >= 50,000',     len(df_ml) >= 50000,                          len(df_ml)),
    # Null check split: numeric columns vs label/string columns
    ('No null values (numeric)',    df_ml.select_dtypes(include='number').isnull().sum().sum() == 0,
                                    df_ml.select_dtypes(include='number').isnull().sum().sum()),
    ('No null values (str cols)',   df_ml[['pressure_label','alignment_label',
                                           'goal','edge_case_type']].isnull().sum().sum() == 0,
                                    df_ml[['pressure_label','alignment_label',
                                           'goal','edge_case_type']].isnull().sum().sum()),
    ('Study hours in [0,10]',       df_ml['study_hours_day'].between(0,10).all(), '✓'),
    ('Sleep hours in [4,9]',        df_ml['sleep_hours_day'].between(4,9).all(),  '✓'),
    ('Screen time in [0,8]',        df_ml['screen_time_day'].between(0,8).all(),  '✓'),
    ('Attendance in [60,100]',      df_ml['attendance_pct'].between(60,100).all(),'✓'),
    ('Performance in [0,100]',      df_ml['performance_score'].between(0,100).all(),'✓'),
    ('Pressure score in [0,1]',     df_ml['pressure_score'].between(0,1).all(),   '✓'),
    ('High pressure records > 5%',  (df_ml['pressure_label']=='High').mean() > 0.05, f"{(df_ml['pressure_label']=='High').mean()*100:.1f}%"),
    ('Edge case % in [6,7]',        6.0 <= edge_pct <= 7.0,                       f'{edge_pct:.2f}%'),
    ('All 10 edge types present',   df_ml['edge_case_type'].nunique() == 11,       df_ml['edge_case_type'].nunique()),  # 10 types + 'normal'
    ('All 4 goals present',         df_ml['goal'].nunique() == 4,                  df_ml['goal'].value_counts().to_dict()),
    ('All 3 pressure labels',       df_ml['pressure_label'].nunique() == 3,        df_ml['pressure_label'].value_counts().to_dict()),
    ('20 weeks per student',        df_ml['week'].nunique() == 20,                 df_ml['week'].nunique()),
    ('14 feature columns',          len(FEATURE_COLS) == 14,                       len(FEATURE_COLS)),
]

all_pass = True
for name, passed, detail in checks:
    status = '✅ PASS' if passed else '❌ FAIL'
    if not passed: all_pass = False
    print(f'  {status}  {name:<40} {detail}')

print('=' * 62)
verdict = '✅ ALL CHECKS PASSED — Dataset v3.0 is ready.' if all_pass else '❌ Some checks failed.'
print(f'  RESULT: {verdict}')
print('=' * 62)